# Learn Docker by building a complete ML application

Project: **customer churn prediction API + dashboard + PostgreSQL**.  The dataset is synthetic. No real-life customer decisions should be made using this educational model.

**Recommended:** Read this notebook in Jupyter or VS Code while running Docker commands in an integrated terminal. Docker CLI commands shown as `bash` blocks are intended for the terminal. Notebook Python cells work without Docker only after installing local dependencies.

## Step 1: Understand the architecture

![Architecture](../assets/architecture.png)

**Data flow**: Dashboard sends an HTTP POST to API → API creates a Pandas DataFrame → scikit-learn Pipeline generates a probability → SQLAlchemy stores the prediction in PostgreSQL → API replies to dashboard.

**Container boundaries**: `dashboard` talks to `api` by its Compose service name `api`. `api` talks to Postgres using hostname `db`. Do not use `localhost` between containers.

## Step 2: Inspect the different files

| Artifact | Why it exists |
|---|---|
| Python modules | Train model, run API and dashboard |
| Dockerfiles | Package services into images |
| YAML | Define 3-service application and GitHub Actions CI |
| .env.example | Safely document runtime settings |
| SQLAlchemy model | Persist predictions in DB |
| Tests | Catch ML and API regressions |
| Model artifacts | Serialized trained model and evaluation JSON |
| Makefile | Shortcut commands |
| README | Setup + operations documentation |

Run the next cell from the project root.

In [ ]:
from pathlib import Path
root = Path.cwd()
print("Working directory:", root)
print("Top-level files:", sorted(p.name for p in root.iterdir())[:30])

## Step 3: Train and inspect the ML model

Run in terminal first:

```bash
python -m venv .venv
# Linux/macOS: source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
pip install -r requirements-dev.txt
python -m ml.train
```

The training function synthesizes customer records, splits them into train/test, standardizes numeric columns, one-hot encodes categories, trains logistic regression and writes `artifacts/model.joblib` and `artifacts/metrics.json`.

**Do not load arbitrary joblib files**. Pickle-derived formats can execute malicious code.

In [ ]:
from ml.train import make_dataset, train
import pandas as pd
sample = make_dataset(n=10)
display(sample)
metrics = train()
metrics

## Step 4: Inspect the input feature schema

The inference API uses Pydantic to reject invalid requests, such as negative tenure or unsupported contract types. The `Pipeline` handles preprocessing exactly as during training.

In [ ]:
from app.schemas import CustomerInput
customer = CustomerInput(tenure_months=12, monthly_spend=80, support_tickets=3, contract_type="monthly", auto_pay="no")
customer.model_dump()

## Step 5: Build Docker images

From project root, **in a terminal**:

```bash
cp .env.example .env  # on Windows PowerShell: Copy-Item .env.example .env
# Edit .env and change the example password
docker compose config
docker compose build
```

Dockerfile `trainer` stage trains model. Runtime stage has API source, dependencies, and trained artifact but not training source. Streamlit has its own Dockerfile.

## Step 6: Start the entire platform

```bash
docker compose up -d
docker compose ps
docker compose logs -f api
```

- Dashboard: http://localhost:8501
- API docs: http://localhost:8000/docs
- Model info: http://localhost:8000/model/info

The Postgres service is not exposed on a host port.

## Step 7: Call the API and see persistent history

Use the following code cell after containers start.

In [ ]:
import requests
payload = {"tenure_months": 12, "monthly_spend": 80, "support_tickets": 3,
           "contract_type": "monthly", "auto_pay": "no"}
result = requests.post("http://localhost:8000/predict", json=payload, timeout=10)
result.raise_for_status()
print("Prediction:",result.json())
history = requests.get("http://localhost:8000/predictions", timeout=10)
history.raise_for_status()
display(pd.DataFrame(history.json()).head())

## Step 8: Understand how persistence works

```bash
docker compose exec db psql -U mluser -d mldb -c "SELECT * FROM predictions LIMIT 5;"
docker compose down
docker compose up -d
```

The predictions remain, because Postgres files reside in a named volume.

⚠️ `docker compose down -v` **deletes** that database volume.

## Step 9: Debug running containers

```bash
docker compose ps
docker compose logs --tail=50 api
docker compose exec api sh
docker compose exec api env
docker inspect $(docker compose ps -q api)
docker stats
```

The API listens on `0.0.0.0:8000` inside the container and the host publishes it at `localhost:8000`. Publishing a port and `EXPOSE` are not the same thing.

## Step 10: CI/CD and further engineering

Check `.github/workflows/ci.yml`. PRs run automated tests and build images. Pushes to `main` publish a commit-tagged API image to GHCR, **not automatically deploy** it.

### Exercises

1. Add a new input feature and adapt dataset, schema, and frontend consistently.
2. Add `GET /predictions/{id}` with a test.
3. Replace the synthetic generator with a carefully versioned real dataset and evaluate across meaningful customer segments.
4. Add Alembic DB migrations and a Postgres-backed integration test.
5. Add an API authentication mechanism and restrict predictions access.
6. Add an explicit, approval-gated deployment stage.

**Success criteria:** `pytest -q` passes; the Compose stack starts; `/predict` returns a probability; `/predictions` persists across `docker compose down` and `up`.